# Panda Baseline Retrain — Kaggle Notebook A

**Purpose:** Retrain the 21M Panda model (predict mode) from scratch on the original training dataset (`GilpinLab/skew40`). This is the control run for the Koopman lifting ablation (Notebook B).

**Config:** Matches `GilpinLab/panda` (21M checkpoint) — d_model=512, 8 layers, 100k steps, `use_dynamics_embedding=True`.

**Output:** Checkpoint saved to `/kaggle/working/checkpoint-final/` — download and use locally for evaluation.

**Notebook B diff:** Change `USE_DYNAMICS_EMBEDDING = True` → `False` and `RUN_NAME = 'baseline'` → `'koopman_ablation'`. Everything else identical.

## 0. Ablation flag — only line that differs between Notebook A and B

In [ ]:
# ============================================================
# ABLATION FLAG — change this for Notebook B
# Notebook A (baseline):         USE_DYNAMICS_EMBEDDING = True
# Notebook B (Koopman ablation): USE_DYNAMICS_EMBEDDING = False
# ============================================================
USE_DYNAMICS_EMBEDDING = True
RUN_NAME = 'baseline'  # change to 'koopman_ablation' for Notebook B

## 1. Install dependencies

In [ ]:
# Install panda repo (architecture + training code)
!git clone --depth=1 https://github.com/abao1999/panda.git

# Install panda dependencies
# Note: panda uses uv but we install manually for Kaggle compatibility
%cd panda
!pip install -e . --quiet

# Additional dependencies needed for training
!pip install gluonts wandb --quiet

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
        print(f'  VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')
        print(f'  Compute capability: {torch.cuda.get_device_capability(i)}')

## 2. Load training data from HuggingFace

In [ ]:
from datasets import load_dataset
import numpy as np

print('Downloading GilpinLab/skew40 (~3GB)...')
hf_dataset = load_dataset('GilpinLab/skew40', split='train')
print(f'Loaded {len(hf_dataset)} trajectories')
print(f'Columns: {hf_dataset.column_names}')

# Inspect one example to verify format
example = hf_dataset[0]
target = np.array(example['target'])
print(f'\nExample trajectory shape: {target.shape}')  # expect [C, T]
print(f'Start: {example["start"]}')
print(f'Source dir: {example["_source_directory"]}')

In [ ]:
# Wrapper: makes HF dataset look like a gluonts Dataset
# gluonts expects an iterable of dicts with keys 'start' and 'target'
# target shape: [C, T] (multivariate)

from gluonts.dataset.common import Dataset as GluonTSDataset
import pandas as pd

class HFSkew40Dataset(GluonTSDataset):
    """
    Wraps GilpinLab/skew40 HuggingFace dataset as a gluonts Dataset.
    Yields dicts with 'start' (pd.Period) and 'target' (np.ndarray [C, T]).
    """
    def __init__(self, hf_dataset, freq='h'):
        self.hf_dataset = hf_dataset
        self.freq = freq

    def __iter__(self):
        for row in self.hf_dataset:
            target = np.array(row['target'], dtype=np.float32)
            # HF stores target as flat list with shape metadata
            # reshape using stored shape if needed
            if 'target._np_shape' in row and row['target._np_shape'] is not None:
                shape = row['target._np_shape']
                target = target.reshape(shape)
            # gluonts expects pd.Period for start
            start = pd.Period(row['start'], freq=self.freq)
            yield {'start': start, 'target': target}

    def __len__(self):
        return len(self.hf_dataset)


# Verify the wrapper works
wrapper = HFSkew40Dataset(hf_dataset)
test_item = next(iter(wrapper))
print(f'Wrapper test — target shape: {test_item["target"].shape}, start: {test_item["start"]}')
print('Wrapper OK')

## 3. Model config — 21M baseline

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/panda')

from transformers import PatchTSTConfig
from panda.patchtst.patchtst import PatchTSTForPrediction
from panda.utils.train_utils import load_patchtst_model

# 21M model config — matches GilpinLab/panda checkpoint
# use_dynamics_embedding controlled by ablation flag above
MODEL_CONFIG = dict(
    mode='predict',
    context_length=512,
    prediction_length=128,
    patch_length=16,
    patch_stride=16,
    num_hidden_layers=8,
    d_model=512,
    num_attention_heads=8,
    channel_attention=True,
    ffn_dim=512,
    norm_type='rmsnorm',
    norm_eps=1e-5,
    attention_dropout=0.0,
    positional_dropout=0.0,
    path_dropout=0.0,
    ff_dropout=0.0,
    bias=True,
    activation_function='gelu',
    pre_norm=True,
    use_cls_token=False,
    init_std=0.02,
    scaling='std',
    pooling_type='max',
    head_dropout=0.0,
    # rope
    channel_rope=False,
    max_wavelength=500,
    rope_percent=0.75,
    # loss
    loss='mse',
    distribution_output=None,
    # Koopman lifting — controlled by ablation flag
    use_dynamics_embedding=USE_DYNAMICS_EMBEDDING,
    num_poly_feats=120,
    poly_degrees=2,
    rff_trainable=False,
    rff_scale=1.0,
    num_rff=256,
    # masking (unused in predict mode but required by config)
    do_mask_input=None,
    mask_type='random',
    random_mask_ratio=0.5,
    channel_consistent_masking=False,
    mask_value=0,
    num_forecast_mask_patches=3,
    unmasked_channel_indices=None,
    num_parallel_samples=100,
)

model = load_patchtst_model(
    mode='predict',
    model_config=MODEL_CONFIG,
    pretrained_encoder_path=None,
    pretained_checkpoint=None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: use_dynamics_embedding={USE_DYNAMICS_EMBEDDING}')
print(f'Trainable parameters: {trainable_params:,}')
# Expect ~21M for baseline, slightly less for ablation (no RFF/poly weights)

## 4. Build training dataset

In [ ]:
import torch
import transformers
from functools import partial

from panda.dataset import MultivariateTimeSeriesDataset
from panda.augmentations import (
    RandomAffineTransform,
    RandomConvexCombinationTransform,
    RandomDimSelectionTransform,
    RandomFourierSeries,
    RandomPhaseSurrogate,
    RandomTakensEmbedding,
    StandardizeTransform,
)

SEED = 99
transformers.set_seed(SEED)

# Augmentations — same as original training
# probabilities from dataset.yaml: [1.0, 1.0, 1.0, 0.0, 0.0]
# (RandomFourierSeries and RandomPhaseSurrogate disabled)
augmentations = [
    RandomTakensEmbedding(lag_range=[1, 10], random_seed=SEED),
    RandomConvexCombinationTransform(alpha=1.0, random_seed=SEED, dim_range=[3, 8]),
    RandomAffineTransform(dim_range=[3, 8], scale=1.0, random_seed=SEED),
    RandomPhaseSurrogate(cutoff=1.0, random_seed=SEED),
    RandomFourierSeries(max_wavenumber=10.0, max_amp=10.0, mode_range=[5, 15], random_seed=SEED),
]
aug_probs_raw = [1.0, 1.0, 1.0, 0.0, 0.0]
aug_probs = [p / sum(aug_probs_raw) for p in aug_probs_raw]  # normalise to sum=1

transforms = [
    StandardizeTransform(),
    RandomDimSelectionTransform(num_dims=3),  # fixed_dim=3 from dataset.yaml
]

# Build dataset from HF wrapper
gluon_dataset = HFSkew40Dataset(hf_dataset)

train_dataset = MultivariateTimeSeriesDataset(
    datasets=[gluon_dataset],
    probabilities=[1.0],
    context_length=512,
    prediction_length=128,
    mode='train',
    model_type='predict',
    augmentations=augmentations,
    augmentation_probabilities=aug_probs,
    augmentation_rate=0.2,
    transforms=transforms,
).shuffle(shuffle_buffer_length=10_000)

# Smoke test — check one batch comes through
test_iter = iter(train_dataset)
test_batch = next(test_iter)
print(f'past_values shape: {test_batch["past_values"].shape}')    # expect [512, 3]
print(f'future_values shape: {test_batch["future_values"].shape}') # expect [128, 3]
print('Dataset OK')

## 5. Training

In [ ]:
from transformers.training_args import TrainingArguments
from transformers.trainer import Trainer
from panda.utils.train_utils import ensure_contiguous

OUTPUT_DIR = f'/kaggle/working/{RUN_NAME}'

# T4 has 16GB VRAM — batch 512 will OOM
# Use gradient accumulation: effective batch = 32 * 16 = 512
PER_DEVICE_BATCH = 32
GRAD_ACCUM = 16  # effective batch size = 512
MAX_STEPS = 100_000

# TF32 requires compute capability >= 8.0; T4 is 7.5 — disabled automatically
# torch_compile should work on T4 (CUDA 7.5)

training_args = TrainingArguments(
    run_name=RUN_NAME,
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=1e-3,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    weight_decay=0.0,
    optim='adamw_torch',  # fused version may not be available on T4
    logging_strategy='steps',
    logging_steps=500,
    save_strategy='steps',
    save_steps=25_000,   # 4 intermediate checkpoints
    max_steps=MAX_STEPS,
    dataloader_num_workers=2,  # Kaggle has limited CPU cores
    remove_unused_columns=False,
    report_to=[],  # disable wandb/tensorboard on Kaggle
    seed=SEED,
    fp16=True,  # use fp16 on T4 (no bf16 support)
    torch_compile=False,  # disable initially — enable if stable
    ddp_find_unused_parameters=False,
)

ensure_contiguous(model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print(f'Run: {RUN_NAME}')
print(f'use_dynamics_embedding: {USE_DYNAMICS_EMBEDDING}')
print(f'Effective batch size: {PER_DEVICE_BATCH * GRAD_ACCUM * max(1, torch.cuda.device_count())}')
print(f'Max steps: {MAX_STEPS}')
print(f'Output: {OUTPUT_DIR}')
print('Starting training...')

trainer.train()

## 6. Save final checkpoint

In [ ]:
import json, os
from omegaconf import OmegaConf

final_ckpt_dir = os.path.join(OUTPUT_DIR, 'checkpoint-final')
model.save_pretrained(final_ckpt_dir)

# Save training info so we can load correctly during evaluation
training_info = {
    'run_name': RUN_NAME,
    'use_dynamics_embedding': USE_DYNAMICS_EMBEDDING,
    'max_steps': MAX_STEPS,
    'model_config': MODEL_CONFIG,
    'train_config': {
        'per_device_train_batch_size': PER_DEVICE_BATCH,
        'gradient_accumulation_steps': GRAD_ACCUM,
        'learning_rate': 1e-3,
        'seed': SEED,
    }
}
with open(os.path.join(final_ckpt_dir, 'training_info.json'), 'w') as f:
    json.dump(training_info, f, indent=2)

print(f'Checkpoint saved to: {final_ckpt_dir}')
print('Download checkpoint-final/ directory for local evaluation.')

## 7. Quick in-distribution sanity check

Before downloading, verify the checkpoint produces sensible forecasts on one held-out trajectory from the test split. This is not the full evaluation — just a sanity check that training converged.

In [ ]:
from datasets import load_dataset as hf_load
from panda.patchtst.pipeline import PatchTSTPipeline
import numpy as np

# Load test split
hf_test = hf_load('GilpinLab/skew40', split='test')
test_example = hf_test[0]
target = np.array(test_example['target'])
if 'target._np_shape' in test_example and test_example['target._np_shape'] is not None:
    target = target.reshape(test_example['target._np_shape'])

# target shape: [C, T] — take first 512 as context
C, T = target.shape
context = target[:, :512].T  # [512, C] — pipeline expects [T, C]
context_tensor = torch.tensor(context, dtype=torch.float32)

# Load from saved checkpoint
pipeline = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path=final_ckpt_dir,
    device_map='cpu',
)

pred = pipeline.predict(context_tensor, 128, limit_prediction_length=False)
pred = pred.squeeze().cpu().numpy()

# Ground truth for comparison
truth = target[:, 512:640].T  # [128, C]

# Compute MAE as quick sanity metric
# Per-window normalise (same as our evaluation protocol)
ctx_mean = context.mean(axis=0, keepdims=True)
ctx_std = context.std(axis=0, keepdims=True) + 1e-8
pred_norm = (pred - ctx_mean) / ctx_std
truth_norm = (truth - ctx_mean) / ctx_std
mae = np.mean(np.abs(pred_norm - truth_norm))

print(f'Sanity check MAE (normalised, one test trajectory): {mae:.4f}')
print('If MAE < 1.0, training likely converged. If >> 1.0, check training logs.')